# MEC_53614 — Artificial intelligence for solids and structures — Martin Genet & Ioannis Stefanou

# L3 — Identifying fields and failure modes, with uncertainty

# E3 — Stiffness field of a bar, and failure modes of columns

In [ ]:
# Names: …

As usual, we start with some imports…

In [ ]:
# Standard python libraries
import warnings

# Computing libraries
import numpy as np    # doc: https://numpy.org/doc/stable/reference
import sklearn as skl # doc: https://scikit-learn.org/stable/api/index.html

# Plotting libraries
import matplotlib.pyplot as mpl # doc: https://matplotlib.org/stable/contents.html

# Course library (on Colab, uncomment the following line first)
# !pip install git+https://github.com/mgenet/LIB53614.git
import LIB53614 as lib

## Problem

**Parts A–D.** A polymer bar has been produced by additive manufacturing (3D printing), and is suspected to contain a zone of lack-of-fusion porosity, i.e., a zone of lower stiffness.
The bar, of length $\ell = 100$ mm and cross-section area $S = 20$ mm², is clamped at $x = 0$ and pulled by a force $N = 200$ N at $x = \ell$.
Its displacement field $u(x)$ is measured by digital image correlation (DIC) at 101 equally spaced points, with a noise of about 0.5 µm.

We want to identify the Young modulus field $E(x)$—where is the defect, and how soft is it?—and, as importantly, to know where this identification can be trusted.

Since the stress $\sigma = N/S$ is uniform, the strain is $\varepsilon(x) = \sigma c(x)$, where $c = 1/E$ is the compliance, and
$$
    u(x) = \sigma \int_0^x c(s) \, ds.
$$
As in E1, the data are synthetic, and the truth can be retrieved with `lib.bar.truth(bar)`, only to evaluate our methods.

**Part E.** The second problem, a classification problem on steel columns, is introduced in part E.

## A. Data and naive inversion

In [ ]:
bar = lib.bar.specimen(seed=0)
sigma = bar.force / bar.area
x = bar.x; u = bar.u; n = len(x)
h = x[1] - x[0]                # distance between measurement points (mm)
x_c = 0.5 * (x[1:] + x[:-1])   # centers of the intervals between measurement points (mm)
E_true = lib.bar.truth(bar)["E"](x_c)

mpl.plot(x, u, "+")
mpl.xlabel("x (mm)")
mpl.ylabel("u (mm)")
mpl.grid();

**Q1. Can you see the defect on the measured displacement field?**

**Q2. The simplest approach is to compute the strain by finite differences, $\varepsilon_{i+1/2} = \left(u_{i+1} - u_i\right)/h$, and then $E_{i+1/2} = \sigma / \varepsilon_{i+1/2}$.
Complete and execute the following code.
What is the root-mean-square (RMS) relative error on $E$?
Why is it so large, while the displacement noise is only 0.5 µm?**

Hints:
* `np.diff(V)` returns the differences between consecutive components of `V`.
* The RMS of `V` is `np.sqrt(np.mean(V**2))`.

In [ ]:
def rms_relative_error(E, E_true):
    return ### YOUR CODE HERE ###

eps_naive = ### YOUR CODE HERE ###
E_naive = ### YOUR CODE HERE ###
print(f"RMS relative error: {100*rms_relative_error(E_naive, E_true):.1f}%")

mpl.plot(x_c, E_naive, ".", label="naive")
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

**Q3. In practice, DIC software smooths the strain over a window.
Complete and execute the following code, which averages the finite-difference strains over windows of $k$ intervals, and computes the corresponding error.
Which window gives the smallest error? What limits the window from below, and from above?
(Compare with the choice of the fitting window in E1.)**

Hint:
* `np.convolve(V, np.ones(k)/k, mode="same")` returns the moving average of `V` over `k` components.

In [ ]:
for k in [1, 3, 5, 9, 15, 25, 51]:
    eps_smoothed = ### YOUR CODE HERE ###
    E_smoothed = sigma / eps_smoothed
    print(f"k = {k:2d}: RMS relative error = {100*rms_relative_error(E_smoothed[k//2:len(x_c)-k//2], E_true[k//2:len(x_c)-k//2]):5.1f}% (away from the ends)")

## B. A linear inverse problem, and its regularization

We now discretize the compliance as piecewise constant on the $n - 1$ intervals between measurement points: $c_j$ on $\left[x_j, x_{j+1}\right]$.
The displacement at the measurement points then reads $u_i = \sigma h \sum_{j < i} c_j$, i.e., $\underline{\mathbb{U}} = \underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}}$, where $\underline{\underline{\mathbb{H}}}$ is a known $n \times (n-1)$ matrix.
The identification of $\underline{\mathbb{C}}$ from the measured $\underline{\mathbb{U}}$ is a linear inverse problem.

**Q4. Complete and execute the following code, which builds $\underline{\underline{\mathbb{H}}}$ and checks it on the true compliance.
What is the RMS of the residual $\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}}^{true} - \underline{\mathbb{U}}$? Is that what you expect?**

Hint:
* `np.tril(np.ones((n, m)), -1)` returns the $n \times m$ matrix with ones strictly below the diagonal, and zeros elsewhere.

In [ ]:
H = ### YOUR CODE HERE ###
c_true = 1. / E_true
print(f"RMS of the residual: {np.sqrt(np.mean((H @ c_true - u)**2)):.2e} mm (noise: {bar.noise:.1e} mm)")

**Q5. Complete and execute the following code, which solves the inverse problem in the least-squares sense, without regularization.
Compare with the naive approach of Q2. Why?**

Hint:
* `np.linalg.lstsq(M, V)[0]` returns the least-squares solution of `M.X = V`.

In [ ]:
c_ls = ### YOUR CODE HERE ###
print(f"RMS relative error: {100*rms_relative_error(1./c_ls, E_true):.1f}% (naive: {100*rms_relative_error(E_naive, E_true):.1f}%)")
print(f"condition number of H: {np.linalg.cond(H):.0f}")

The inverse problem is ill-posed: the noise is amplified by the "inversion" of $\underline{\underline{\mathbb{H}}}$, i.e., by differentiation.
The classical remedy is Tikhonov regularization: we look for the compliance that minimizes
$$
    J_\lambda\left(\underline{\mathbb{C}}\right) = \left\|\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}} - \underline{\mathbb{U}}\right\|^2 + \lambda \left\|\underline{\underline{\mathbb{D}}} \cdot \underline{\mathbb{C}}\right\|^2,
$$
where $\underline{\underline{\mathbb{D}}}$ is the finite-difference matrix, $\left(\underline{\underline{\mathbb{D}}} \cdot \underline{\mathbb{C}}\right)_j = c_{j+1} - c_j$, and $\lambda > 0$ the regularization parameter.

**Q6. Show that the minimizer is the solution of $\left(\underline{\underline{\mathbb{H}}}^T \cdot \underline{\underline{\mathbb{H}}} + \lambda \underline{\underline{\mathbb{D}}}^T \cdot \underline{\underline{\mathbb{D}}}\right) \cdot \underline{\mathbb{C}} = \underline{\underline{\mathbb{H}}}^T \cdot \underline{\mathbb{U}}$.**

**Q7. Complete and execute the following code, which computes the Tikhonov solution for a range of $\lambda$, and plots both the error (with respect to the truth) and the norm of the residual $\left\|\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}}_\lambda - \underline{\mathbb{U}}\right\|$.
Which $\lambda$ gives the smallest error, and what is this error?
Without knowing the truth, the "discrepancy principle" selects the largest $\lambda$ for which the residual is at the noise level, i.e., $\left\|\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}}_\lambda - \underline{\mathbb{U}}\right\| \leq \sqrt{n} \, \sigma_u$: which $\lambda$ does it select, and with which error?**

Hints:
* `np.diff(np.eye(m), axis=0)` returns the $(m-1) \times m$ finite-difference matrix.
* `np.linalg.solve(M, V)` returns the solution of `M.X = V`.

In [ ]:
D = ### YOUR CODE HERE ###

def tikhonov(H, u, D, lmbda):
    return ### YOUR CODE HERE ###

lmbda_lst = np.logspace(0, 8, 33)
error_lst = []
residual_lst = []
for lmbda in lmbda_lst:
    c = tikhonov(H, u, D, lmbda)
    ### YOUR CODE HERE ###
    ### YOUR CODE HERE ###
error_lst = np.array(error_lst); residual_lst = np.array(residual_lst)

k_best = np.argmin(error_lst)
k_disc = ### YOUR CODE HERE ###
print(f"smallest error:        lambda = {lmbda_lst[k_best]:.1e}, RMS relative error = {100*error_lst[k_best]:.2f}%")
print(f"discrepancy principle: lambda = {lmbda_lst[k_disc]:.1e}, RMS relative error = {100*error_lst[k_disc]:.2f}%")

fig, axes = mpl.subplots(1, 2, figsize=(14, 5))
axes[0].loglog(lmbda_lst, error_lst)
axes[0].set_xlabel("λ"); axes[0].set_ylabel("RMS relative error on E"); axes[0].grid()
axes[1].loglog(lmbda_lst, residual_lst)
axes[1].axhline(np.sqrt(n) * bar.noise, color="k", ls="--", label="noise level")
axes[1].set_xlabel("λ"); axes[1].set_ylabel("norm of the residual (mm)"); axes[1].grid(); axes[1].legend();

**Q8. Plot the Tikhonov solution for the $\lambda$ selected by the discrepancy principle, and for a 100 times larger $\lambda$.
What does the regularization term assume about the field? What happens to the defect when $\lambda$ is too large?**

In [ ]:
for lmbda in [lmbda_lst[k_disc], 100 * lmbda_lst[k_disc]]:
    ### YOUR CODE HERE ###
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

## C. The Bayesian view: Gaussian processes

We now consider the compliance field as a random field, with a Gaussian-process prior: $c(x) \sim \mathcal{GP}\left(m, k\right)$, with a constant mean $m$ and the squared-exponential covariance
$$
    k\left(x, x'\right) = s^2 \exp\left(-\frac{\left(x - x'\right)^2}{2 l^2}\right),
$$
where $s$ is the standard deviation of the field and $l$ its correlation length.
On the intervals, this gives $\underline{\mathbb{C}} \sim \mathcal{N}\left(m \underline{\mathbb{1}}, \underline{\underline{\mathbb{K}}}\right)$, with $\mathbb{K}_{ij} = k\left(x_i, x_j\right)$.
The measurements are $\underline{\mathbb{U}} = \underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}} + \underline{\mathbb{e}}$, with a Gaussian noise $\underline{\mathbb{e}} \sim \mathcal{N}\left(\underline{\mathbb{0}}, \sigma_u^2 \underline{\underline{\mathbb{1}}}\right)$.
Since everything is Gaussian and linear, the posterior (i.e., the distribution of $\underline{\mathbb{C}}$ given the measurements) is also Gaussian, with mean and covariance
$$
    \underline{\mathbb{C}}^{post} = m \underline{\mathbb{1}} + \underline{\underline{\mathbb{K}}} \cdot \underline{\underline{\mathbb{H}}}^T \cdot \underline{\underline{\mathbb{S}}}^{-1} \cdot \left(\underline{\mathbb{U}} - m \underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{1}}\right),
    \quad
    \underline{\underline{\mathbb{K}}}^{post} = \underline{\underline{\mathbb{K}}} - \underline{\underline{\mathbb{K}}} \cdot \underline{\underline{\mathbb{H}}}^T \cdot \underline{\underline{\mathbb{S}}}^{-1} \cdot \underline{\underline{\mathbb{H}}} \cdot \underline{\underline{\mathbb{K}}},
    \quad \text{with} \quad
    \underline{\underline{\mathbb{S}}} = \underline{\underline{\mathbb{H}}} \cdot \underline{\underline{\mathbb{K}}} \cdot \underline{\underline{\mathbb{H}}}^T + \sigma_u^2 \underline{\underline{\mathbb{1}}}.
$$

**Q9. Show that the posterior mean is also the minimizer of
$\left\|\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}} - \underline{\mathbb{U}}\right\|^2 / \sigma_u^2 + \left(\underline{\mathbb{C}} - m \underline{\mathbb{1}}\right)^T \cdot \underline{\underline{\mathbb{K}}}^{-1} \cdot \left(\underline{\mathbb{C}} - m \underline{\mathbb{1}}\right)$.
Which prior corresponds to the Tikhonov regularization of Q7?**

Hint:
* For any matrices of compatible sizes, $\left(\underline{\underline{\mathbb{H}}}^T \cdot \underline{\underline{\mathbb{H}}} / \sigma_u^2 + \underline{\underline{\mathbb{K}}}^{-1}\right)^{-1} \cdot \underline{\underline{\mathbb{H}}}^T / \sigma_u^2 = \underline{\underline{\mathbb{K}}} \cdot \underline{\underline{\mathbb{H}}}^T \cdot \left(\underline{\underline{\mathbb{H}}} \cdot \underline{\underline{\mathbb{K}}} \cdot \underline{\underline{\mathbb{H}}}^T + \sigma_u^2 \underline{\underline{\mathbb{1}}}\right)^{-1}$.

**Q10. Before using the measurements, let us look at the prior.
Complete and execute the following code, which draws samples of the prior for three correlation lengths.
Which one seems compatible with a lack-of-fusion defect of a few millimeters?
Does the true field look like a typical sample of any of them?**

Hints:
* `np.exp(V)` acts componentwise, and `V[:,None] - V[None,:]` returns the matrix of the differences between all the components of `V`.
* `rng.multivariate_normal(mean, cov, size=k)` returns `k` samples of the multivariate normal distribution of mean `mean` and covariance `cov`.

In [ ]:
def kernel(x1, x2, s, l):
    return ### YOUR CODE HERE ###

m = u[-1] / (sigma * bar.length) # mean compliance of the bar, from its end displacement
rng = np.random.default_rng(0)
fig, axes = mpl.subplots(1, 3, figsize=(18, 4), sharey=True)
for ax, l in zip(axes, [2., 10., 30.]):
    K = kernel(x_c, x_c, s=0.1*m, l=l) + 1e-8 * m**2 * np.eye(len(x_c)) # (small "jitter" for numerical stability)
    samples = ### YOUR CODE HERE ###
    for sample in samples:
        ax.plot(x_c, 1./sample, lw=1)
    ax.plot(x_c, E_true, "k", lw=2, label="truth")
    ax.set_title(f"prior samples, s = 0.1 m, l = {l:.0f} mm")
    ax.set_xlabel("x (mm)")
    ax.grid()
axes[0].set_ylabel("E (MPa)")
axes[0].legend();

**Q11. Complete and execute the following code, which computes the posterior mean and covariance for given hyperparameters $\left(m, s, l\right)$, and plots the posterior mean of $E$ with its 95% credible band.
We take for $m$ the mean compliance of the bar, given by its end displacement, and, for now, $s = 0.1 m$ and $l = 10$ mm.
What is the error? How does it compare with Tikhonov?**

Hints:
* `np.linalg.solve(M, V)` also works when `V` is a matrix.
* `np.diag(M)` returns the diagonal of `M`.
* Since $E = 1/c$ is a decreasing function of $c$, the 95% credible band of $E$ is $\left[1/\left(c^{post} + 1.96 \, \sigma^{post}\right), 1/\left(c^{post} - 1.96 \, \sigma^{post}\right)\right]$.

In [ ]:
def gp_posterior(H, u, noise, x_c, m, s, l):
    K = kernel(x_c, x_c, s, l)
    S = ### YOUR CODE HERE ###
    mean = ### YOUR CODE HERE ###
    cov  = ### YOUR CODE HERE ###
    return mean, cov

def plot_posterior(x_c, mean, cov, label):
    std = np.sqrt(np.diag(cov))
    line = mpl.plot(x_c, 1./mean, label=label)[0]
    mpl.fill_between(x_c, 1./(mean + 1.96*std), 1./(mean - 1.96*std), color=line.get_color(), alpha=0.3)

c_post, K_post = gp_posterior(H, u, bar.noise, x_c, m=m, s=0.1*m, l=10.)
print(f"RMS relative error: {100*rms_relative_error(1./c_post, E_true):.2f}%")

plot_posterior(x_c, c_post, K_post, label="GP posterior (s = 0.1 m, l = 10 mm)")
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

The hyperparameters $\left(s, l\right)$ play the role of $\lambda$.
They can be chosen by maximizing the probability of the measurements given the hyperparameters, i.e., the marginal likelihood:
$$
    \log p\left(\underline{\mathbb{U}} | s, l\right) = - \frac{1}{2} \underline{\mathbb{r}}^T \cdot \underline{\underline{\mathbb{S}}}^{-1} \cdot \underline{\mathbb{r}} - \frac{1}{2} \log \det \underline{\underline{\mathbb{S}}} - \frac{n}{2} \log\left(2 \pi\right),
    \quad \text{with} \quad
    \underline{\mathbb{r}} = \underline{\mathbb{U}} - m \underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{1}}.
$$

**Q12. Complete and execute the following code, which maximizes the marginal likelihood over a grid of hyperparameters.
Which hyperparameters do you get? Compare $l$ with the true width of the defect, $w$ (in `lib.bar.truth(bar)`).
What is now the error?**

Hint:
* `np.linalg.slogdet(M)[1]` returns $\log \left|\det M\right|$.

In [ ]:
def log_marginal_likelihood(H, u, noise, x_c, m, s, l):
    K = kernel(x_c, x_c, s, l)
    S = H @ K @ H.T + noise**2 * np.eye(len(u))
    r = u - m * H @ np.ones(len(x_c))
    return ### YOUR CODE HERE ###

s_lst = m * np.geomspace(0.01, 1., 21)
l_lst = np.geomspace(1., 50., 21)
logml = np.array([[log_marginal_likelihood(H, u, bar.noise, x_c, m, s, l) for l in l_lst] for s in s_lst])
k_s, k_l = np.unravel_index(np.argmax(logml), logml.shape)
s_opt, l_opt = s_lst[k_s], l_lst[k_l]
print(f"optimal hyperparameters: s = {s_opt/m:.3f} m, l = {l_opt:.1f} mm (true defect width: w = {lib.bar.truth(bar)['w']:.1f} mm)")

c_post, K_post = gp_posterior(H, u, bar.noise, x_c, m, s_opt, l_opt)
print(f"RMS relative error: {100*rms_relative_error(1./c_post, E_true):.2f}%")

mpl.contourf(l_lst, s_lst/m, logml, levels=np.linspace(logml.max()-50, logml.max(), 26), extend="min")
mpl.plot(l_opt, s_opt/m, "r+", ms=15)
mpl.xscale("log"); mpl.yscale("log")
mpl.xlabel("l (mm)"); mpl.ylabel("s / m")
mpl.colorbar(label="log marginal likelihood");

**Q13. Complete and execute the following code, which draws samples of the posterior obtained with the optimal hyperparameters.
What does each sample represent?
How well is the stiffness at the center of the defect determined?**

In [ ]:
samples = ### YOUR CODE HERE ###
for sample in samples:
    mpl.plot(x_c, 1./sample, lw=1)
plot_posterior(x_c, c_post, K_post, label="posterior mean and 95% band")
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

print("minimum of E in each sample:", np.round(1./samples.max(axis=1)), f"MPa (truth: {E_true.min():.0f} MPa)")

**Q14. Complete and execute the following code, which compares the posteriors obtained with the optimal hyperparameters and with $l = 30$ mm, and computes the fraction of intervals where the truth lies within the 95% credible band (the "coverage").
What do you observe? What does it tell you about credible bands?**

In [ ]:
for l, label in [(l_opt, f"l = {l_opt:.1f} mm (optimal)"), (30., "l = 30 mm")]:
    c_post, K_post = gp_posterior(H, u, bar.noise, x_c, m, s_opt, l)
    std = np.sqrt(np.diag(K_post))
    coverage = ### YOUR CODE HERE ###
    print(f"{label}: RMS relative error = {100*rms_relative_error(1./c_post, E_true):.2f}%, coverage = {100*coverage:.0f}%")
    plot_posterior(x_c, c_post, K_post, label=label)
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

## D. Where can the identification be trusted?

**Q15. Execute the following code, which plots the relative posterior standard deviation along the bar.
Where is the identification the least certain? Why?**

In [ ]:
c_post, K_post = gp_posterior(H, u, bar.noise, x_c, m, s_opt, l_opt)
mpl.plot(x_c, np.sqrt(np.diag(K_post)) / c_post)
mpl.xlabel("x (mm)")
mpl.ylabel("relative posterior standard deviation")
mpl.grid();

**Q16. Looking at the formulas, does the posterior covariance depend on the measured values?
Complete and execute the following code, which checks it by replacing the measurements by zeros.
What does it imply for the design of a DIC setup?**

In [ ]:
_, K_post_0 = ### YOUR CODE HERE ###
print("same posterior covariance:", np.allclose(K_post_0, K_post))

## E. Failure modes of columns: from regression to classification

Our second problem is a classification problem.
A database gathers compression tests on pin-ended steel columns of nominal grade S355 (yield stress $\sigma_y = 355$ MPa, Young modulus $E = 210$ GPa).
The columns have various slenderness ratios $\lambda = L/i$ ($L$: length; $i$: radius of gyration of the cross-section), and are loaded at various mean stresses $\sigma = P/A$.
Each test is labelled by its outcome: 0 = "safe" (the column carries the load), 1 = "plastic collapse", 2 = "elastic buckling".

We want to predict the outcome for a new column and a new load, and to compare our predictions with the proven methods: the theory of perfect columns (squash load and Euler critical stress), and the buckling curves of the Eurocode 3 (EC3).

As before, the data are synthetic: the yield stress and the geometric imperfection vary from column to column (cf. `help(lib.column)`), and the truth is only used to evaluate our methods.
We use 300 tests for training, and 2000 other tests to evaluate the predictions.

In [ ]:
train = lib.column.campaign(n=300, seed=0)
test  = lib.column.campaign(n=2000, seed=1)
E_col, sigma_y = lib.column.E_NOMINAL, lib.column.SIGMA_Y_NOMINAL # nominal values (MPa)
print("classes:", lib.column.CLASSES)
print("number of training tests per class:", np.bincount(train.label))

class_colors = ["tab:green", "tab:orange", "tab:red"]
def plot_campaign(ax, campaign, s=12, alpha=1.):
    for k, name in enumerate(lib.column.CLASSES):
        ax.scatter(campaign.lmbda[campaign.label == k], campaign.sigma[campaign.label == k], s=s, color=class_colors[k], alpha=alpha, label=name)
    ax.set_xlabel("slenderness λ = L/i")
    ax.set_ylabel("applied stress σ = P/A (MPa)")
    ax.grid(True, which="both", alpha=0.3)

fig, axes = mpl.subplots(1, 2, figsize=(14, 5))
plot_campaign(axes[0], train)
plot_campaign(axes[1], train)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[0].legend();

**Q17. According to the theory of perfect columns, a column fails by plastic collapse when $\sigma$ reaches the yield stress $\sigma_y$ (squash load), and by elastic buckling when $\sigma$ reaches the Euler stress $\sigma_E = \pi^2 E / \lambda^2$, whichever comes first.
Show that, in the $\left(\log \lambda, \log \sigma\right)$ plane, the boundaries between the three classes are straight lines; give their slopes, and the slenderness $\lambda_c$ at which they meet.
Complete and execute the following code, which draws them on the data.
Do they separate the classes? Where are the exceptions?**

In [ ]:
lmbda_plot = np.geomspace(10., 300., 200)
lmbda_c = ### YOUR CODE HERE ###
print(f"lambda_c = {lmbda_c:.1f}")

def plot_perfect_column(ax):
    ax.plot(lmbda_plot[lmbda_plot < lmbda_c], np.full(np.sum(lmbda_plot < lmbda_c), sigma_y), "k--", lw=1.5) # squash load
    ### YOUR CODE HERE ### # Euler stress
    ax.plot([lmbda_c, lmbda_c], [sigma_y, 2000.], "k--", lw=1.5) # boundary between the two failure modes
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(12., 270.); ax.set_ylim(18., 900.)

fig, ax = mpl.subplots(figsize=(7, 5))
plot_campaign(ax, train)
plot_perfect_column(ax)
ax.legend();

### E.1. Failure or not? Logistic regression

We first ignore the failure mode, and only predict whether the column fails ($y = 1$) or not ($y = 0$).
The logistic regression model gives the probability of failure as
$$
    p\left(\underline{x}\right) = s\left(\underline{\tilde{x}} \cdot \underline{w}\right),
    \quad \text{with} \quad
    s(z) = \frac{1}{1 + e^{-z}}
$$
the sigmoid function, $\underline{\tilde{x}} = \left(1, z_1, z_2\right)$, and $\left(z_1, z_2\right)$ the standardized (zero mean, unit standard deviation over the training set) features $\left(\log \lambda, \log \sigma\right)$.
Since each label follows a Bernoulli distribution of parameter $p$, the negative log-likelihood of the training labels, divided by $N$, is the cross-entropy:
$$
    L\left(\underline{w}\right) = -\frac{1}{N} \sum_{i=1}^{N} \left[y_i \log p_i + \left(1 - y_i\right) \log\left(1 - p_i\right)\right],
    \quad \text{with} \quad
    p_i = s\left(\underline{\tilde{x}}_i \cdot \underline{w}\right).
$$

**Q18. Show that $s'(z) = s(z) \left(1 - s(z)\right)$, and that the gradient of the cross-entropy reads $\underline{\nabla} L = \underline{\underline{\mathbb{X}}}^T \cdot \left(\underline{\mathbb{P}} - \underline{\mathbb{Y}}\right) / N$, where $\underline{\underline{\mathbb{X}}}$ is the design matrix (one row $\underline{\tilde{x}}_i$ per test), $\underline{\mathbb{P}} = \left(p_i\right)_i$ and $\underline{\mathbb{Y}} = \left(y_i\right)_i$.
Compare with the gradient of the least-squares loss of linear regression (cf. L2/E2).**

**Q19. Complete and execute the following code, which trains the logistic regression by gradient descent (as in E2), after checking the gradient by finite differences.
What is the accuracy (fraction of correct predictions) on the test set?
Compare your weights with those of scikit-learn.**

Hints:
* Use the training mean and standard deviation to standardize the features of both the training and test sets.
* If the gradient check fails (relative difference much larger than $10^{-6}$), fix the gradient before training!
* `np.log(V + 1e-12)` avoids $\log 0$.

In [ ]:
def features(campaign):
    return np.stack([np.log(campaign.lmbda), np.log(campaign.sigma)], axis=1)

F_train = features(train)
F_mean, F_std = F_train.mean(axis=0), F_train.std(axis=0)
def design_matrix(campaign):
    Z = ### YOUR CODE HERE ###
    return np.hstack([np.ones((len(Z), 1)), Z])

X_train, X_test = design_matrix(train), design_matrix(test)
y_train, y_test = (train.label > 0).astype(float), (test.label > 0).astype(float)

def sigmoid(z):
    return ### YOUR CODE HERE ###

def cross_entropy(w, X, y):
    p = sigmoid(X @ w)
    return ### YOUR CODE HERE ###

def gradient(w, X, y):
    return ### YOUR CODE HERE ###

# Gradient check, at a random point
w_rand = np.random.default_rng(0).standard_normal(3)
g_fd = np.array([(cross_entropy(w_rand + 1e-6*e, X_train, y_train) - cross_entropy(w_rand - 1e-6*e, X_train, y_train)) / 2e-6 for e in np.eye(3)])
print(f"gradient check: relative difference = {np.linalg.norm(gradient(w_rand, X_train, y_train) - g_fd) / np.linalg.norm(g_fd):.1e}")

# Gradient descent
w = np.zeros(3)
learning_rate = 1.
loss_history = []
for iteration in range(5000):
    w = ### YOUR CODE HERE ###
    loss_history.append(cross_entropy(w, X_train, y_train))

accuracy = ### YOUR CODE HERE ###
print(f"weights: {w}, final loss: {loss_history[-1]:.4f}, test accuracy: {accuracy:.3f}")

model = skl.linear_model.LogisticRegression(C=np.inf).fit(X_train[:,1:], y_train) # C = inf: no regularization
print(f"scikit-learn: weights: {np.append(model.intercept_, model.coef_)}, test accuracy: {model.score(X_test[:,1:], y_test):.3f}")

mpl.semilogy(loss_history)
mpl.xlabel("iteration"); mpl.ylabel("cross-entropy"); mpl.grid();

**Q20. Execute the following code, which plots the contours $p = 0.05$, $0.5$ and $0.95$ of the predicted probability of failure, on top of the test data and of the theory of perfect columns.
Where are the errors?
Why can a logistic regression on these features not separate the failed columns from the others, while the perfect-column boundary is made of straight lines?**

In [ ]:
log_lmbda_grid, log_sigma_grid = np.meshgrid(np.log(np.geomspace(12., 270., 200)), np.log(np.geomspace(18., 900., 200)))
Z_grid = (np.stack([log_lmbda_grid.ravel(), log_sigma_grid.ravel()], axis=1) - F_mean) / F_std
X_grid = np.hstack([np.ones((len(Z_grid), 1)), Z_grid])

fig, ax = mpl.subplots(figsize=(7, 5))
plot_campaign(ax, test, s=4, alpha=0.5)
plot_perfect_column(ax)
contours = ax.contour(np.exp(log_lmbda_grid), np.exp(log_sigma_grid), sigmoid(X_grid @ w).reshape(log_lmbda_grid.shape), levels=[0.05, 0.5, 0.95], colors="b")
ax.clabel(contours)
ax.legend();

### E.2. Which failure mode? Softmax regression

We now predict the three classes.
The softmax (or multinomial logistic) regression has one weight vector per class, gathered in a $3 \times K$ matrix $\underline{\underline{\mathbb{W}}}$ (here $K = 3$ classes), and gives the probabilities of the classes as
$$
    p_k\left(\underline{x}\right) = \frac{\exp\left(\underline{\tilde{x}} \cdot \underline{w}_k\right)}{\sum_{l=1}^{K} \exp\left(\underline{\tilde{x}} \cdot \underline{w}_l\right)}.
$$
With the "one-hot" encoding of the labels, $Y_{ik} = 1$ if test $i$ belongs to class $k$ and 0 otherwise, the cross-entropy reads $L = -\frac{1}{N} \sum_{i} \sum_{k} Y_{ik} \log P_{ik}$, and one can show (as in the binary case) that its gradient is $\underline{\underline{\mathbb{X}}}^T \cdot \left(\underline{\underline{\mathbb{P}}} - \underline{\underline{\mathbb{Y}}}\right) / N$.

**Q21. Complete and execute the following code, which trains the softmax regression by gradient descent, and plots the predicted classes.
What is the test accuracy?
Merging the two failure modes, what is its accuracy for predicting failure, compared with the logistic regression of Q19? Why is it better?**

Hints:
* `np.eye(K)[labels]` returns the one-hot encoding of an array of integer labels.
* Since $\exp$ quickly overflows, compute the softmax of $\underline{\tilde{x}} \cdot \underline{w}_k - \max_l \underline{\tilde{x}} \cdot \underline{w}_l$, which gives the same probabilities. `Z.max(axis=1, keepdims=True)` returns the maximum of each row of `Z`, as a column.

In [ ]:
K = 3
Y_train = ### YOUR CODE HERE ###

def softmax(Z):
    Z = ### YOUR CODE HERE ###
    return ### YOUR CODE HERE ###

def cross_entropy_softmax(W, X, Y):
    return ### YOUR CODE HERE ###

def gradient_softmax(W, X, Y):
    return ### YOUR CODE HERE ###

# Gradient check
W_rand = np.random.default_rng(0).standard_normal((3, K))
G_fd = np.zeros((3, K))
for i in range(3):
    for k in range(K):
        dW = np.zeros((3, K)); dW[i,k] = 1e-6
        G_fd[i,k] = (cross_entropy_softmax(W_rand + dW, X_train, Y_train) - cross_entropy_softmax(W_rand - dW, X_train, Y_train)) / 2e-6
print(f"gradient check: relative difference = {np.linalg.norm(gradient_softmax(W_rand, X_train, Y_train) - G_fd) / np.linalg.norm(G_fd):.1e}")

# Gradient descent
W = np.zeros((3, K))
for iteration in range(20000):
    W = ### YOUR CODE HERE ###

label_pred = ### YOUR CODE HERE ###
accuracy_failure = ### YOUR CODE HERE ###
print(f"test accuracy (3 classes): {np.mean(label_pred == test.label):.3f}")
print(f"test accuracy (failure or not): {accuracy_failure:.3f} (logistic regression: {accuracy:.3f})")

fig, ax = mpl.subplots(figsize=(7, 5))
ax.contourf(np.exp(log_lmbda_grid), np.exp(log_sigma_grid), np.argmax(X_grid @ W, axis=1).reshape(log_lmbda_grid.shape), levels=[-0.5, 0.5, 1.5, 2.5], colors=class_colors, alpha=0.2)
plot_campaign(ax, test, s=4, alpha=0.5)
plot_perfect_column(ax)
ax.legend();

**Q22. The boundary between classes $k$ and $l$ is where their probabilities are equal, i.e., where $\underline{\tilde{x}} \cdot \left(\underline{w}_k - \underline{w}_l\right) = 0$: a straight line in the $\left(z_1, z_2\right)$ plane, hence in the $\left(\log \lambda, \log \sigma\right)$ plane.
Complete and execute the following code, which computes the slopes $d \log \sigma / d \log \lambda$ of the three learned boundaries.
Compare with the theory of perfect columns (Q17).**

Hint:
* Since $z_j = \left(\log x_j - \mu_j\right) / s_j$, a line $a_0 + a_1 z_1 + a_2 z_2 = 0$ has slope $-\left(a_1 / s_1\right) / \left(a_2 / s_2\right)$ in the $\left(\log \lambda, \log \sigma\right)$ plane.

In [ ]:
for k, l in [(0, 1), (0, 2), (1, 2)]:
    a = W[:,k] - W[:,l]
    slope = ### YOUR CODE HERE ###
    print(f"boundary between {lib.column.CLASSES[k]:16s} and {lib.column.CLASSES[l]:16s}: slope = {slope:6.2f}")

**Q23. Complete and execute the following code, which trains the same softmax regression (with scikit-learn) on the raw features $\left(\lambda, \sigma\right)$, standardized, instead of their logarithms.
What is the test accuracy? Why? (Compare with E1, part D.)**

In [ ]:
def raw_features(campaign):
    return np.stack([campaign.lmbda, campaign.sigma], axis=1)

F_raw_mean, F_raw_std = raw_features(train).mean(axis=0), raw_features(train).std(axis=0)
model_raw = ### YOUR CODE HERE ###
accuracy_raw = ### YOUR CODE HERE ###
print(f"test accuracy (3 classes, raw features): {accuracy_raw:.3f}")

**Q24. Complete and execute the following code, which computes the confusion matrix of the softmax regression of Q21 on the test set (row: true class; column: predicted class), and plots the misclassified tests.
Which confusions occur, and where? Why there?**

Hint:
* `skl.metrics.confusion_matrix(y_true, y_pred)` returns the confusion matrix.

In [ ]:
confusion = ### YOUR CODE HERE ###
print(confusion)

wrong = ### YOUR CODE HERE ###
fig, ax = mpl.subplots(figsize=(7, 5))
plot_campaign(ax, test, s=4, alpha=0.2)
ax.scatter(test.lmbda[wrong], test.sigma[wrong], s=40, facecolors="none", edgecolors="k", label="misclassified")
plot_perfect_column(ax)
ax.legend();

### E.3. Comparison with the proven methods

**Q25. Complete and execute the following code, which computes the test accuracy of the perfect-column rule (Q17, with the nominal values), and which compares the test data with the design buckling stress of the EC3 (curve b, without partial safety factor), $\sigma_{EC3}(\lambda)$ = `lib.column.ec3_stress(lmbda)`.
Which fraction of the failed columns were loaded above $\sigma_{EC3}$? Which fraction of the safe columns?
Which approach would you use to design a column, and why?**

In [ ]:
sigma_E_test = lib.column.euler_stress(test.lmbda)
label_perfect = ### YOUR CODE HERE ###
print(f"test accuracy of the perfect-column rule: {np.mean(label_perfect == test.label):.3f}")

sigma_ec3 = lib.column.ec3_stress(test.lmbda)
failed = test.label > 0
fraction_failed_above = ### YOUR CODE HERE ###
fraction_safe_above = ### YOUR CODE HERE ###
print(f"failed columns loaded above the EC3 stress: {100*fraction_failed_above:.1f}%")
print(f"safe columns loaded above the EC3 stress:   {100*fraction_safe_above:.1f}%")

fig, ax = mpl.subplots(figsize=(7, 5))
plot_campaign(ax, test, s=4, alpha=0.5)
plot_perfect_column(ax)
ax.plot(lmbda_plot, lib.column.ec3_stress(lmbda_plot), "b", lw=2, label="EC3, curve b")
ax.legend();

**Q26. Complete and execute the following code, which compares, for a column of slenderness $\lambda = 80$, the probability of failure predicted by the softmax regression (one minus the probability of "safe") with the true probability that a column of the production fails (`lib.column.failure_probability`, for checking only), for several applied stresses.
Is the probability given by the classifier a probability of failure? What would be needed to estimate one, e.g., for a reliability analysis?**

In [ ]:
for sigma_applied in [180., 200., 210., 220., 240.]:
    x_tilde = np.hstack([1., (np.log([80., sigma_applied]) - F_mean) / F_std])
    p_fail = ### YOUR CODE HERE ###
    print(f"sigma = {sigma_applied:.0f} MPa: predicted P(failure) = {p_fail:.3f}, true P(failure) = {lib.column.failure_probability(80., sigma_applied):.3f}, EC3 stress = {lib.column.ec3_stress(80.):.0f} MPa")

## Bonus

The following questions are optional. They continue parts B, D and C, and apply Gaussian processes to the surrogate modeling of an expensive simulation.

**Q27. (Continuation of part B.) Instead of the gradient of the compliance, we could penalize the compliance itself, as in ridge regression (cf. L2), i.e., minimize $\left\|\underline{\underline{\mathbb{H}}} \cdot \underline{\mathbb{C}} - \underline{\mathbb{U}}\right\|^2 + \lambda \left\|\underline{\mathbb{C}} - c_0 \underline{\mathbb{1}}\right\|^2$.
Complete and execute the following code, which does it for $c_0 = 0$, and for $c_0$ equal to the mean compliance of the bar (given by its end displacement), with $\lambda$ chosen by the discrepancy principle.
Which errors do you get? What does each penalty assume about the field?**

In [ ]:
I = np.eye(n-1)
for name, c_0 in [("c_0 = 0", 0.), ("c_0 = mean compliance", u[-1] / (sigma * bar.length))]:
    error_lst_ridge = []
    residual_lst_ridge = []
    for lmbda in lmbda_lst:
        c = ### YOUR CODE HERE ###
        error_lst_ridge.append(rms_relative_error(1./c, E_true))
        residual_lst_ridge.append(np.linalg.norm(H @ c - u))
    k = np.flatnonzero(np.array(residual_lst_ridge) <= np.sqrt(n) * bar.noise)[-1]
    print(f"identity penalty, {name:22s}: lambda = {lmbda_lst[k]:.1e}, RMS relative error = {100*error_lst_ridge[k]:.2f}%")

**Q28. (Continuation of part D.) A cheaper DIC setup gives only 26 measurement points, with a noise of 2 µm.
Complete and execute the following code, which repeats the identification on such data (with the hyperparameters found above).
How do the error and the credible band change? Is the defect still detected?**

In [ ]:
bar_c = lib.bar.specimen(seed=0, n_points=26, noise=2e-3)
x_c_c = 0.5 * (bar_c.x[1:] + bar_c.x[:-1])
H_c = ### YOUR CODE HERE ###
c_post_c, K_post_c = ### YOUR CODE HERE ###
print(f"RMS relative error: {100*rms_relative_error(1./c_post_c, lib.bar.truth(bar_c)['E'](x_c_c)):.2f}%")

plot_posterior(x_c, c_post, K_post, label="101 points, 0.5 µm")
plot_posterior(x_c_c, c_post_c, K_post_c, label="26 points, 2 µm")
mpl.plot(x_c, E_true, "k", label="truth")
mpl.xlabel("x (mm)")
mpl.ylabel("E (MPa)")
mpl.grid()
mpl.legend();

**Q29. (Continuation of part D.) Three other bars from the same production have been measured with the same setup.
Complete and execute the following code, which identifies their stiffness fields with the hyperparameters found on the first bar (keeping the same ratio $s/m$).
Can the hyperparameters be transferred from one bar to another? What do they characterize: a bar, or the production process?**

In [ ]:
for seed in [1, 2, 3]:
    bar_k = lib.bar.specimen(seed=seed)
    m_k = bar_k.u[-1] / (sigma * bar_k.length)
    c_post_k, K_post_k = ### YOUR CODE HERE ###
    E_true_k = lib.bar.truth(bar_k)["E"](x_c)
    coverage_k = np.mean(np.abs(1./E_true_k - c_post_k) <= 1.96 * np.sqrt(np.diag(K_post_k)))
    print(f"bar {seed}: defect width w = {lib.bar.truth(bar_k)['w']:.1f} mm, RMS relative error = {100*rms_relative_error(1./c_post_k, E_true_k):.2f}%, coverage = {100*coverage_k:.0f}%")

### A surrogate of an expensive simulation: imperfection-sensitive buckling

Thin structures under compression (launcher stages, silos, cans…) buckle at loads much lower than their theoretical critical load, because of small geometric imperfections.
Design practice still relies largely on empirical "knock-down factors" (e.g., NASA SP-8007), which are conservative.
Computing the actual limit load requires an expensive nonlinear simulation for each imperfection.

We consider a simple model of such a structure (a rigid bar, pinned at its base and held at its top by a softening lateral spring, under an axial load).
Its limit load $\lambda_{max}$ (normalized by the critical load of the perfect structure) depends on the amplitude $\theta_0$ of its initial imperfection, and is computed by `lib.buckling.limit_load(theta_0)`.
Each call plays the role of an expensive simulation, and is counted in `lib.buckling.n_calls`.
For small imperfections, Koiter's asymptotic theory predicts $\lambda_{max} \approx 1 - a \, \theta_0^{2/3}$ for this type of structure (with a coefficient $a$ that depends on the structure).

The imperfections of the production are known from measurements: $\theta_0$ follows a log-normal distribution, with a median of 0.005 rad and a standard deviation of $\log \theta_0$ of 0.5.
We want the probability that the limit load is lower than the design load $\lambda_d = 0.85$, with as few simulations as possible.

In [ ]:
# Reference curve, for checking only (in practice, we could not afford it)
theta_ref = np.linspace(0., 0.04, 201)
lmbda_ref = np.array([lib.buckling.limit_load(theta) for theta in theta_ref])
lib.buckling.reset_counter()

def fit_gp(X, y):
    kernel = skl.gaussian_process.kernels.ConstantKernel(1., (1e-3, 1e3)) * skl.gaussian_process.kernels.RBF(0.5, (1e-3, 1e3))
    with warnings.catch_warnings(): # (the hyperparameter optimizer may complain when the data are fitted perfectly)
        warnings.simplefilter("ignore")
        return skl.gaussian_process.GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True, n_restarts_optimizer=5, random_state=0).fit(X, y)

**Q30. Complete and execute the following code, which builds a Gaussian-process surrogate of $\lambda_{max}\left(\theta_0\right)$ from 5 simulations, equally spaced on $\left[0, 0.04\right]$ rad, and plots it with its 95% credible band.
What is the maximum error? Are the credible bands reliable? Where, and why?**

Hint:
* `gp.predict(X, return_std=True)` returns the posterior mean and standard deviation of the GP `gp` at the points `X` (a 2D array, one column per input).

In [ ]:
def features(theta):
    return theta[:,None] / 0.04 # (scaled input)

theta_train = np.linspace(0., 0.04, 5)
lmbda_train = ### YOUR CODE HERE ###
gp = fit_gp(features(theta_train), lmbda_train)
mean, std = ### YOUR CODE HERE ###
print(f"maximum error: {np.max(np.abs(mean - lmbda_ref)):.4f}, maximum standard deviation: {np.max(std):.4f}, coverage: {100*np.mean(np.abs(mean - lmbda_ref) <= 1.96*std):.0f}%")

mpl.plot(theta_ref, lmbda_ref, "k", label="reference")
mpl.plot(theta_ref, mean, label="GP surrogate")
mpl.fill_between(theta_ref, mean - 1.96*std, mean + 1.96*std, alpha=0.3)
mpl.plot(theta_train, lmbda_train, "o", label="simulations")
mpl.xlabel("θ_0 (rad)")
mpl.ylabel("λ_max")
mpl.grid()
mpl.legend();

**Q31. Use Koiter's asymptotic result to choose a better input feature for the Gaussian process, and repeat Q30 with the same 5 simulations.
What is now the maximum error? Why is it so much better?**

In [ ]:
### YOUR CODE HERE ###

gp = fit_gp(features(theta_train), lmbda_train)
mean, std = gp.predict(features(theta_ref), return_std=True)
print(f"maximum error: {np.max(np.abs(mean - lmbda_ref)):.4f}, maximum standard deviation: {np.max(std):.4f}, coverage: {100*np.mean(np.abs(mean - lmbda_ref) <= 1.96*std):.0f}%")

mpl.plot(theta_ref, lmbda_ref, "k", label="reference")
mpl.plot(theta_ref, mean, label="GP surrogate (Koiter feature)")
mpl.fill_between(theta_ref, mean - 1.96*std, mean + 1.96*std, alpha=0.3)
mpl.plot(theta_train, lmbda_train, "o", label="simulations")
mpl.xlabel("θ_0 (rad)")
mpl.ylabel("λ_max")
mpl.grid()
mpl.legend();

**Q32. Active learning: complete and execute the following code, which starts from 3 simulations ($\theta_0 = 0$, 0.02 and 0.04 rad), and adds simulations one at a time where the predictive standard deviation is the largest, until it is smaller than $10^{-3}$ everywhere.
How many simulations are needed, and what is the final maximum error, with the raw input, and with the Koiter feature?
Can the standard deviation of the GP be trusted as a stopping criterion?**

In [ ]:
for name, features_fn in [("raw input", lambda theta: theta[:,None] / 0.04), ("Koiter feature", features)]: # (features: your function above, with the Koiter feature)
    lib.buckling.reset_counter()
    theta_train = np.array([0., 0.02, 0.04])
    lmbda_train = np.array([lib.buckling.limit_load(theta) for theta in theta_train])
    while True:
        gp = fit_gp(features_fn(theta_train), lmbda_train)
        mean, std = gp.predict(features_fn(theta_ref), return_std=True)
        if (np.max(std) < 1e-3) or (lib.buckling.n_calls >= 30): break
        theta_new = ### YOUR CODE HERE ###
        theta_train = ### YOUR CODE HERE ###
        lmbda_train = ### YOUR CODE HERE ###
    print(f"{name:14s}: {lib.buckling.n_calls:2d} simulations, maximum error: {np.max(np.abs(mean - lmbda_ref)):.4f}")

**Q33. Complete and execute the following code, which estimates the probability that $\lambda_{max} < \lambda_d$ by Monte Carlo sampling of the imperfections, using the last surrogate (and, for checking only, the reference curve).
Compare with the knock-down-factor approach, which would design the structure with the limit load of the 99th-percentile imperfection.
How many simulations did you use in total?**

Hint:
* `np.exp(np.log(a) + b * rng.standard_normal(N))` returns `N` samples of a log-normal distribution of median `a` and log-standard deviation `b`.

In [ ]:
rng = np.random.default_rng(0)
theta_samples = ### YOUR CODE HERE ###
theta_samples = np.clip(theta_samples, 0., 0.04) # (the surrogate is only valid on [0, 0.04])
lmbda_samples = ### YOUR CODE HERE ###
print(f"P(lambda_max < 0.85): {np.mean(lmbda_samples < 0.85):.4f} (surrogate), {np.mean(np.interp(theta_samples, theta_ref, lmbda_ref) < 0.85):.4f} (reference)")
theta_99 = np.exp(np.log(0.005) + 0.5 * 2.326)
print(f"99th-percentile imperfection: {theta_99:.4f} rad, limit load: {gp.predict(features_fn(np.array([theta_99])))[0]:.3f}")

**Q34. Without a surrogate, how many simulations would a plain Monte Carlo estimation of this probability require, for a relative standard deviation of 10%?**

Hint:
* If $N_f$ of the $N$ samples fail, the estimator $\hat{p} = N_f / N$ has a variance $p \left(1 - p\right) / N$.

## AI audit

Please indicate in the following cell (the format of this audit will be detailed soon):
* which AI tools you used, and for what;
* every error you found in their outputs, and how you found it (comparison with the truth, a plot, units, a limiting case, a test…);
* what you would ask, or forbid, an assistant next time to avoid these errors.

In [ ]:
# AI audit: …